In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import layers


from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator

2024-03-18 14:59:21.172844: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-18 14:59:21.849159: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-03-18 14:59:24.259295: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 14:59:24.281845: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 14:59:24.281885: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 14:59:24.399913: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-03-18 14:59:24.399972: I external/local_xla/xla/stream_executor

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 2843733673912001926
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 12946505404264343432
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [3]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ'}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [42]:
def get_dataset_fromCsv(path):
    cnt = 0
    print("Helelo?")
    
    csvFile = open(path, 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in tqdm(reader):
        imgFile = line[0]

        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        
        
        def dataset_generator():
            #labels = np.array([int(line[1]), int(line[2]), int(line[3])], dtype= 'int64')
            # print(cnt)
            label1 = np.expand_dims(np.array(int(line[1])), axis=0)
            label2 = np.expand_dims(np.array(int(line[2])), axis=0)
            label3 = np.expand_dims(np.array(int(line[3])), axis=0)

            yield img, (label1, label2, label3)

        #                              args=("/root/Data/hangul/dataset/tranDataset.csv",))
        
    dataset = tf.data.Dataset.from_generator(dataset_generator,
                                                output_signature=
                                                (
                                                tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                                (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                                tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                                tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                                )
                                                #output_shapes=([19,21,28])
                                                )

        #dataset = dataset.batch(16).take(10)
        
        
    return dataset
           
        
        
    
#get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv")

In [43]:
dataset =  get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv")
print(dataset)
iterator = iter(dataset)
print(next(iterator))

Helelo?


KeyboardInterrupt: 

In [ ]:

posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

x = layers.Flatten()(posts_input)

DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='DenseCho2')(DenseCho)
DenseJung = layers.Dense(21, activation='softmax', name='DenseJung2')(DenseJung)
DenseJong = layers.Dense(28, activation='softmax', name='DenseJong2')(DenseJong)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

# model.compile(loss=losses, optimizer='adam', metrics=[['accuracy'], ['accuracy'], ['accuracy']])
# model.compile(loss=["sparse_categorical_crossentropy", "sparse_categorical_crossentropy", "sparse_categorical_crossentropy"], 
#               optimizer='adam', 
#               metrics=[['accuracy'], ['accuracy'], ['accuracy']])


# model.compile(loss=["categorical_crossentropy", "categorical_crossentropy", "categorical_crossentropy"], 
#               optimizer='adam', 
#               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

In [ ]:
#dataset = tf.data.Dataset.from_tensor_slices(({'input_x': data_a, 'input_y': data_b}, labels)).batch(2).repeat()
model.fit(get_dataset_fromCsv("/root/Data/hangul/dataset/tranDataset.csv"), epochs = 100, batch_size = 16)
#model.fit_generator(dataset, epochs = 100)


Helelo?
0
Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 0.0000e+00 - DenseJung2_accuracy: 0.0000e+00 - loss: 7.1680
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 5/100


2024-03-18 15:12:38.473780: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:38.473845: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)
2024-03-18 15:12:38.537727: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:38.537773: W tensorflow/core/framework/local_rendezvous.cc:404] Local rende

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - DenseCho2_

2024-03-18 15:12:38.706540: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:38.706630: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-18 15:12:38.740131: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:38.740188: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-18 15:12:38.770985: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:38.771035: W 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 13/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 14/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 15/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 16/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 17/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - DenseC

...

2024-03-18 15:12:42.650286: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:42.650335: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-18 15:12:42.684004: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:42.684059: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-18 15:12:42.716886: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:42.716934: W 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 92/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 93/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 94/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 95/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 96/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00

2024-03-18 15:12:42.856249: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:42.856295: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-03-18 15:12:42.856308: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 9191991864429175534
2024-03-18 15:12:42.856313: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10220727970903567242
2024-03-18 15:12:42.856318: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 11718925914193467114
2024-03-18 15:12:42.856342: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 14274495685733646192
2024-03-

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 97/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 98/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 99/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00
Epoch 100/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 0.0000e+00


2024-03-18 15:12:43.085009: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:43.085059: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-18 15:12:43.119581: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:43.119629: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-03-18 15:12:43.151856: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-03-18 15:12:43.151902: W 